# Guiding Hand · treino do detector de obstáculos (Kaggle)

Antes de rodar: **Settings → Accelerator: GPU T4 x2** e **Internet: On**. Depois use **Save Version → Save & Run All** para rodar em segundo plano.

Ao final, baixe `resultados.zip` na aba **Output**: ele tem o modelo (`guiding_hand_yolo11.pt`), as métricas e os gráficos do treino.

In [ ]:
import socket

try:
    socket.create_connection(("github.com", 443), timeout=10).close()
except OSError:
    raise RuntimeError("Sem internet: ative Settings → Internet → On (exige celular verificado) e rode de novo.")
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "SEM GPU: ative Settings → Accelerator → GPU T4 x2"

In [ ]:
%cd /kaggle/working
!git clone -q https://github.com/joaos-2/guiding-hand-vision.git
%cd guiding-hand-vision
!pip install -q ultralytics

## 1. Dataset (Open Images V7 + buracos)

In [ ]:
!python scripts/01_download_openimages.py
!python scripts/02_download_potholes.py
!python scripts/03_build_dataset.py

## 2. Treino

`yolo11s` é mais preciso que o `yolo11n` e ainda roda em tempo real. Para usar as duas GPUs, troque `--device 0` por `--device 0,1`.

In [ ]:
MODEL, EPOCHS, NAME = "yolo11s.pt", 80, "gh_y11s"
!python scripts/04_train.py --model {MODEL} --epochs {EPOCHS} --batch 32 --device 0 --cache ram --name {NAME}

## 3. Avaliação no conjunto de teste

In [ ]:
!python scripts/05_evaluate.py --device 0

In [ ]:
import shutil
from IPython.display import Image, display

display(Image(f"runs/{NAME}/results.png", width=900))
display(Image("docs/resultados/exemplos_teste.jpg", width=900))

shutil.copytree(f"runs/{NAME}", "/kaggle/working/resultados/treino", dirs_exist_ok=True,
                ignore=shutil.ignore_patterns("last.pt"))
shutil.copytree("docs/resultados", "/kaggle/working/resultados/avaliacao", dirs_exist_ok=True)
shutil.copy("models/guiding_hand_yolo11.pt", "/kaggle/working/resultados/")
shutil.make_archive("/kaggle/working/resultados", "zip", "/kaggle/working/resultados")

In [ ]:
# Remove o dataset da pasta de saída para o Output do Kaggle ficar só com os resultados.
shutil.rmtree("/kaggle/working/guiding-hand-vision")